# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Microsoft Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **A Microsoft Foundry project** with a deployed chat model (e.g. `gpt-5-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.

In [1]:
%pip install agent-framework azure-ai-projects azure-identity -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

c:\Users\podas\Desktop\Study\LearningAgenticAI\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_skills.py:122: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\podas\Desktop\Study\LearningAgenticAI\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_harness\_file_access.py:602: ExperimentalWarning: [HARNESS] AgentFileStore is experimental and may change or be removed in future versions without notice.


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [3]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [4]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

Great — here are four warm beach destinations from the available list that I recommend, with quick pros/cons so you can pick what fits your vibe.

1. Bali (Indonesia)
- Why: Tropical, warm year-round, iconic beaches, lush scenery, strong surf and great snorkeling/diving.
- Vibe: Laid-back to party-friendly (Seminyak) — lots of wellness/temple culture as well.
- Best time: April–October (dry season).
- Highlights: Uluwatu/Padang Padang beaches, surfing, snorkeling at Nusa Lembongan/Nusa Penida, rice terraces, temples.

2. Rio de Janeiro (Brazil)
- Why: Famous warm beaches (Copacabana, Ipanema), vibrant culture and nightlife.
- Vibe: Energetic, social, great for beach sports and people-watching.
- Best time: December–March (summer) for hottest weather and carnival season.
- Highlights: Beach promenades, sugarloaf/corcovado hikes, samba clubs, boat trips.

3. Barcelona (Spain)
- Why: Mediterranean beaches within a lively city — combine beach time with culture and dining.
- Vibe: Urban bea

## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [6]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

Great choice — Tokyo is a fascinating, endlessly varied city that blends ultra-modern tech and nightlife with deep tradition. Here’s a concise guide to help you decide if it’s the right destination and plan a visit.

Quick snapshot
- Vibe: Fast-paced, safe, ultra-clean, gadget-forward, with pockets of centuries-old tradition.
- Highlights: World-class food scene (from street stalls to Michelin-starred restaurants), neon-lit neighborhoods, historic temples and gardens, cutting-edge museums, anime/pop culture, and efficient public transit.
- Best for: Food lovers, culture seekers, shopping and tech fans, day-trip explorers.

Top things to do
- Historic & spiritual: Senso-ji Temple (Asakusa), Meiji Shrine (Harajuku), Imperial Palace (East Gardens).
- Views & skyline: Tokyo Skytree, Tokyo Tower, Tokyo Metropolitan Government Building (free observation decks).
- Neighborhoods: Shinjuku (transport hub, nightlife), Shibuya (famous crossing, youth culture), Ginza (luxury shopping), Akihabara (

## Summary

In this lesson you learned how to:

- **Create a provider** that connects to Microsoft Foundry Agent Service via `FoundryChatClient`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.